In [ ]:
!pip install selenium
!pip install fake_useragent

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.2/499.2 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 1.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import requests
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver import ActionChains
from fake_useragent import UserAgent
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
import json
import time
from multiprocessing import Pool
import os
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Configuraramos webdriver
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

In [ ]:
# Generamos user agent
ua = UserAgent()
chrome_options.add_argument(f'user-agent={ua.random}')

In [ ]:
# Iniciamos webdriver
driver = webdriver.Chrome(options=chrome_options)
time.sleep(3)

In [ ]:
# Cargamos página de Barcelona en Airbnb
url = "https://www.airbnb.es/s/Barcelona--Spain/homes"
driver.get(url)
time.sleep(3)

In [ ]:
datos_anuncios = []

pagina = 1
while pagina <= 1:
    print(f"Procesando página {pagina}...")
    time.sleep(3)

    # Parsear página de resultados
    soup = BeautifulSoup(driver.page_source, "html.parser")
    listings = soup.select('div.cy5jw6o')
    print(f"🔹 {len(listings)} bloques encontrados")

    for i, listing in enumerate(listings):
        try:
            # --- INFO DESDE RESULTADOS ---
            enlace_a = listing.find_all('a')[0].get('href')
            if enlace_a.startswith("/"):
                enlace_a = "https://www.airbnb.es" + enlace_a

            # 🔹 Extraer ID del anuncio
            id_listing = None
            if "/rooms/" in enlace_a:
                id_listing = enlace_a.split("/rooms/")[1].split("?")[0]

            texto_plano = listing.get_text()
            texto_formateado = listing.get_text(separator='\n', strip=True)
            lineas = texto_formateado.split("\n")

            # --- EXTRAER DATOS CON PALABRAS CLAVE ---
            tipo_alojamiento = None
            descripcion = None
            anfitrion = None
            fechas = None
            precio_noche = None
            precio_total = None
            valoracion = None
            num_noches = None
            mes = None  # <-- nueva variable para mes

            for linea in lineas:
                linea_lower = linea.lower()

                if not tipo_alojamiento and any(palabra in linea for palabra in ["Habitación", "Apartamento", "Estudio"]):
                    tipo_alojamiento = linea

                elif not descripcion and any(palabra in linea for palabra in ["Cama", "Suite", "Dormitorio", "Habitación"]):
                    descripcion = linea

                elif not anfitrion and "anfitrión" in linea_lower:
                    anfitrion = linea

                elif not fechas and re.search(r"\d{1,2}–\d{1,2} \w{3}", linea):
                    fechas = linea

                    # 🔹 Calcular noches y extraer mes
                    match = re.search(r"(\d{1,2})–(\d{1,2}) (\w{3})", fechas)
                    if match:
                        dia_inicio = int(match.group(1))
                        dia_fin = int(match.group(2))
                        num_noches = dia_fin - dia_inicio
                        mes = match.group(3)  # guardamos el mes abreviado

                # Precio total
                if not precio_total and "total" in linea_lower and "€" in linea:
                    match = re.search(r"([\d,.]+)\s?€", linea)
                    if match:
                        precio_total = match.group(1).replace(",", ".") + " €"

                    # Intentar sacar número de noches (texto)
                    noches_match = re.search(r'por (\d+) ?noches', linea_lower)
                    if noches_match:
                        num_noches = int(noches_match.group(1))

                # Precio por noche
                if not precio_noche and "noche" in linea_lower and "€" in linea:
                    match = re.search(r"([\d,.]+)\s?€", linea)
                    if match:
                        precio_noche = match.group(1).replace(",", ".") + " €"

                # Valoración
                if not valoracion:
                    match = re.search(r"\d,\d{1,2} \(\d+\)", linea)
                    if match:
                        valoracion = match.group(0)

            # Calcular precio total si solo tenemos precio por noche y número de noches
            if precio_noche and num_noches and not precio_total:
                valor_noche = float(precio_noche.replace("€","").strip())
                precio_total = "{:.2f} €".format(valor_noche * num_noches)

            # --- ENTRAR AL ANUNCIO ---
            driver.get(enlace_a)
            time.sleep(3)
            soup_detalle = BeautifulSoup(driver.page_source, 'html.parser')
            titulo = soup_detalle.find("div", class_="_14efb46")
            precio = soup_detalle.find("div", class_="_ati8ih")

            # --- GUARDAR DATOS ---
            datos_anuncios.append({
                "pagina": pagina,
                "n_anuncio": i + 1,
                "url": enlace_a,
                "id_listing": id_listing,  # <-- añadido
                "texto_plano_resultados": texto_plano,
                "texto_formateado_resultados": texto_formateado,
                "tipo_alojamiento": tipo_alojamiento,
                "descripcion": descripcion,
                "anfitrion": anfitrion,
                "fechas": fechas,
                "mes": mes,
                "precio_noche": precio_noche,
                "precio_total": precio_total,
                "num_noches": num_noches,
                "valoracion": valoracion,
                "titulo": titulo.get_text(strip=True) if titulo else None,
                "precio": precio.get_text(strip=True) if precio else None
            })

            print(f"✅ Anuncio {i+1} guardado: {titulo.get_text(strip=True) if titulo else 'Sin título'}")

            # Volver atrás
            driver.back()
            time.sleep(3)

        except Exception as e:
            print(f"❌ Error en anuncio {i+1}: {e}")
            continue

    # Pasar a siguiente página
    try:
        siguiente = driver.find_element(By.CSS_SELECTOR, 'a[aria-label="Página siguiente"]')
        driver.execute_script("arguments[0].scrollIntoView(true);", siguiente)
        time.sleep(1)
        ActionChains(driver).move_to_element(siguiente).click().perform()
        time.sleep(3)
        pagina += 1
    except:
        print("✅ No hay más páginas.")
        break


Procesando página 1...
🔹 18 bloques encontrados
✅ Anuncio 1 guardado: Habitación encantadora 1
✅ Anuncio 2 guardado: Habitación en Hospital Sant Pau
✅ Anuncio 3 guardado: Habitación individual con baño privado
✅ Anuncio 4 guardado: Habitación en piso acogedor.
✅ Anuncio 5 guardado: Habitación acogedora bien ubicada
✅ Anuncio 6 guardado: Increíble habitación para una persona
✅ Anuncio 7 guardado: Una habitación chula en BCN
✅ Anuncio 8 guardado: Habitación doble: UPC, ESADE, Barça, Real Club Tennis
✅ Anuncio 9 guardado: Habitación Nou Barris
✅ Anuncio 10 guardado: Habitación en el Eixample. Excelente ubicación.
✅ Anuncio 11 guardado: Habitación para una persona Ramblas/AC*
✅ Anuncio 12 guardado: Habitación mediana, céntrica.
✅ Anuncio 13 guardado: Habitacion luminosa y muy tranquila
✅ Anuncio 14 guardado: Habitación en Barcelona (Eixample-Sant Antoni)
✅ Anuncio 15 guardado: Relájate y Medita en Badalona
✅ Anuncio 16 guardado: Habitacion con balcón en piso vegano Sants Estacio
✅ Anuncio 

In [ ]:
datos_anuncios

[{'pagina': 1,
  'n_anuncio': 1,
  'url': 'https://www.airbnb.es/rooms/948604675243960712?search_mode=regular_search&adults=1&category_tag=Tag%3A8678&check_in=2025-09-11&check_out=2025-09-16&children=0&infants=0&pets=0&photo_id=1713035688&source_impression_id=p3_1755872293_P3OBixe-bWZKLHLb&previous_page_section_name=1000&federated_search_id=ac716984-f1e7-4161-9d6c-a45c1f29cc24',
  'id_listing': '948604675243960712',
  'texto_plano_resultados': '74Evaluaciones4,78EstrellasValoración3Años de experienciaLiaHabitación en Santa Coloma de GramanetQuédate con LiaQuédate con LiaHotel, \xa0·\xa0HotelHabitación encantadora 111–16 sept11–16 septAnfitrión particular, \xa0·\xa0Anfitrión particular198\xa0€Consulta el desglose del precio\xa0198\xa0€ por 5\xa0nochespor 5\xa0nochesValoración media de 4,78 sobre 5, 74\xa0evaluaciones4,78 (74)',
  'texto_formateado_resultados': '74\nEvaluaciones\n4,78\nEstrellas\nValoración\n3\nAños de experiencia\nLia\nHabitación en Santa Coloma de Gramanet\nQuédate con